# rebound_state_factor_target_repair_screen_v1

Notebook-first repair screen for the rebound-state factor BC target path.

- **Canonical baseline**: retained SOXL harness model frontier plus `soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02` as the buy/sell reference baseline.
- **Direct comparison baseline**: `rebound_state_factor_bc_replay_v1`, which had `wf_2020` missing and `wf_2023_2024` rebound/release slices failing.
- **Role**: `diagnose` and unblock the next model-selection step; no actor is retained here.
- **Changed layer**: `model_selection` / target contract stitching / BC sample weighting.
- **Hypothesis**: the existing non-leaking SOXLBacktest `wf_2020` target can be stitched into the rebound-state factor target, and a release-heavy class-balanced BC weighting can fit the sparse rebound/release slices before PPO.
- **Acceptance criteria**: all required walk-forward folds have non-leaking train target rows, selected BC config passes global/fold/slice learnability gates, and BC-only replay metrics are written for the next checkpoint/PPO screen.


In [1]:

from __future__ import annotations

import json
import math
import os
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists():
    raise RuntimeError(f'Could not locate repo root from {Path.cwd()}')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'rebound_state_factor_target_repair_screen_v1'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SOURCE_CARD_NAME = 'rebound_state_factor_bc_replay_v1'
REBOUND_TARGET_CARD_NAME = 'rebound_state_factor_target_contract_v1'
WF2020_TARGET_CARD_NAME = 'cap10_wf2020_soxlbacktest_target_contract_v1'
REFERENCE_EXPERIMENT = 'soxlbacktest_qlib_walkforward_baseline_v1'
REFERENCE_CANDIDATE = 'wf_pass_ytd18_15_15_02'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
REBOUND_TARGET_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / REBOUND_TARGET_CARD_NAME / 'selected_target_contract.csv'
WF2020_TARGET_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / WF2020_TARGET_CARD_NAME / 'soxlbacktest_wf2020_target_contract.csv'
SOURCE_SUMMARY_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / SOURCE_CARD_NAME / 'summary.json'
REBOUND_SUMMARY_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / REBOUND_TARGET_CARD_NAME / 'summary.json'
WF2020_SUMMARY_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / WF2020_TARGET_CARD_NAME / 'summary.json'
REFERENCE_METRICS_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / REFERENCE_EXPERIMENT / 'candidate_metrics.csv'
CARD.mkdir(parents=True, exist_ok=True)

SEED = 20260614
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
GLOBAL_MIN_ACCURACY = 0.70
FOLD_MIN_ACCURACY = 0.70
SLICE_MIN_ACCURACY = 0.60
TARGET_TEST_OVERLAP_MAX = 0

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'target_fold': 'wf_2020_train',
        'train': ('2010-03-11', '2018-12-31'),
        'valid': ('2019-01-01', '2019-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}
REQUIRED_FOLDS = tuple(spec['target_fold'] for spec in WALK_FORWARD_WINDOWS.values())
SLICE_COLS = ('release_gate_state', 'factor_rebound_state', 'is_rebound_cluster', 'is_downtrend_state', 'drawdown_control_state')
# `factor_rebound_state` is absent by construction in the pre-2020 SOXLBacktest target.
# Treat zero-sample optional slices as not applicable, but require accuracy once samples exist.
REQUIRED_SLICE_RULES = {
    'release_gate_state': {'min_samples': 5, 'min_accuracy': SLICE_MIN_ACCURACY, 'required': True},
    'factor_rebound_state': {'min_samples': 5, 'min_accuracy': SLICE_MIN_ACCURACY, 'required': False},
    'is_rebound_cluster': {'min_samples': 5, 'min_accuracy': SLICE_MIN_ACCURACY, 'required': False},
    'is_downtrend_state': {'min_samples': 100, 'min_accuracy': 0.70, 'required': False},
    'drawdown_control_state': {'min_samples': 20, 'min_accuracy': SLICE_MIN_ACCURACY, 'required': False},
}

TRAIN_600 = StateObjectiveBCTrainingConfig(steps=600, lr=0.001, batch_size=128, seed=SEED)
TRAIN_1000 = StateObjectiveBCTrainingConfig(steps=1000, lr=0.001, batch_size=128, seed=SEED)
WEIGHT_CONFIGS = [
    {
        'config_name': 'release10_rebound4_downtrend2_steps600',
        'complexity': 1,
        'description': 'Prior moderate release-gate weighting from cap10 three-fold screen.',
        'train_config': TRAIN_600,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=10.0,
            objective_weight_power=0.0,
            class_balanced_beta=0.0,
            weight_cap=30.0,
        ),
    },
    {
        'config_name': 'release12_rebound4_downtrend2_steps600',
        'complexity': 2,
        'description': 'Slightly stronger release weighting, no class balancing.',
        'train_config': TRAIN_600,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=12.0,
            objective_weight_power=0.0,
            class_balanced_beta=0.0,
            weight_cap=30.0,
        ),
    },
    {
        'config_name': 'release9_rebound4_downtrend2_steps1000',
        'complexity': 3,
        'description': 'Longer optimization sanity check without class balancing.',
        'train_config': TRAIN_1000,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=9.0,
            objective_weight_power=0.0,
            class_balanced_beta=0.0,
            weight_cap=30.0,
        ),
    },
    {
        'config_name': 'release16_rebound6_downtrend2_cb99',
        'complexity': 4,
        'description': 'Release-heavy, rebound-heavy, class-balanced weighting for sparse factor-state repair.',
        'train_config': TRAIN_600,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=6.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=16.0,
            objective_weight_power=0.20,
            class_balanced_beta=0.99,
            weight_cap=35.0,
        ),
    },
]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding='utf-8'))


def normalize_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == object:
        return series.fillna(False).astype(str).str.lower().isin({'true', '1', 'yes', 'y', 't'})
    return series.fillna(False).astype(bool)


def load_contract(path: Path, source_contract_id: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame['datetime'] = pd.to_datetime(frame['datetime']).dt.normalize()
    for col in SLICE_COLS:
        if col in frame.columns:
            frame[col] = normalize_bool_series(frame[col])
        else:
            frame[col] = False
    for col in ('teacher_position', 'rebound_target_position', 'risk_target_position', 'objective_weight'):
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors='coerce')
    frame['source_contract_id'] = source_contract_id
    return frame


def load_reference_metrics(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame = frame[(frame['candidate'] == REFERENCE_CANDIDATE) & (frame['split_type'] == 'walk_forward_test')].copy()
    frame['reference_relative_return_pp'] = frame['rulebase_return_pct'] - frame['soxl_buyhold_return_pct']
    frame['reference_avg_position'] = frame['average_position_pct'] / 100.0
    return frame


def reference_row(reference_metrics: pd.DataFrame, fold: str) -> dict[str, Any]:
    rows = reference_metrics[reference_metrics['window'] == fold]
    if rows.empty:
        return {
            'reference_candidate': REFERENCE_CANDIDATE,
            'reference_return_pct': None,
            'reference_relative_return_pp': None,
            'reference_max_dd_pct': None,
            'reference_avg_position': None,
        }
    row = rows.iloc[0]
    return {
        'reference_candidate': str(row['candidate']),
        'reference_return_pct': float(row['rulebase_return_pct']),
        'reference_relative_return_pp': float(row['reference_relative_return_pp']),
        'reference_max_dd_pct': float(row['rulebase_max_drawdown_pct']),
        'reference_avg_position': float(row['reference_avg_position']),
        'reference_trade_count': int(row['trade_count']),
    }


def target_fold_rows(targets: pd.DataFrame, fold: str, window: dict[str, Any]) -> tuple[pd.DataFrame, dict[str, Any]]:
    train_start, train_end = pd.Timestamp(window['train'][0]), pd.Timestamp(window['train'][1])
    valid_start, valid_end = pd.Timestamp(window['valid'][0]), pd.Timestamp(window['valid'][1])
    test_start, test_end = pd.Timestamp(window['test'][0]), pd.Timestamp(window['test'][1])
    fold_targets = targets[targets['fold'].astype(str).eq(window['target_fold'])].copy()
    train_rows = fold_targets[fold_targets['datetime'].between(train_start, train_end)].copy()
    audit = {
        'fold': fold,
        'target_fold': window['target_fold'],
        'source_contract_ids': ','.join(sorted(fold_targets['source_contract_id'].dropna().astype(str).unique())),
        'fold_rows': int(len(fold_targets)),
        'train_rows': int(len(train_rows)),
        'valid_overlap_rows': int(fold_targets['datetime'].between(valid_start, valid_end).sum()),
        'test_overlap_rows': int(fold_targets['datetime'].between(test_start, test_end).sum()),
        'train_start': window['train'][0],
        'train_end': window['train'][1],
        'valid_start': window['valid'][0],
        'valid_end': window['valid'][1],
        'test_start': window['test'][0],
        'test_end': window['test'][1],
        'release_gate_rows': int(train_rows['release_gate_state'].fillna(False).astype(bool).sum()),
        'factor_rebound_rows': int(train_rows['factor_rebound_state'].fillna(False).astype(bool).sum()),
        'rebound_cluster_rows': int(train_rows['is_rebound_cluster'].fillna(False).astype(bool).sum()),
        'downtrend_rows': int(train_rows['is_downtrend_state'].fillna(False).astype(bool).sum()),
        'drawdown_control_rows': int(train_rows['drawdown_control_state'].fillna(False).astype(bool).sum()),
    }
    audit['target_contract_gate'] = bool(audit['train_rows'] > 0 and audit['test_overlap_rows'] <= TARGET_TEST_OVERLAP_MAX)
    return train_rows, audit


def slice_gate_details(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    details: dict[str, Any] = {}
    gate = True
    for slice_name, rule in REQUIRED_SLICE_RULES.items():
        if slice_name not in indexed.index:
            samples = 0
            accuracy = float('nan')
        else:
            row = indexed.loc[slice_name]
            samples = int(row.get('samples', 0))
            accuracy = float(row.get('accuracy', float('nan'))) if pd.notna(row.get('accuracy', float('nan'))) else float('nan')
        required = bool(rule['required']) or samples > 0
        if not required and samples == 0:
            passed = True
            status = 'not_applicable_zero_samples'
        else:
            passed = bool(samples >= int(rule['min_samples']) and math.isfinite(accuracy) and accuracy >= float(rule['min_accuracy']))
            status = 'passed' if passed else 'failed'
        details[slice_name] = {
            'samples': samples,
            'accuracy': accuracy,
            'min_samples': int(rule['min_samples']),
            'min_accuracy': float(rule['min_accuracy']),
            'required': required,
            'passed': passed,
            'status': status,
        }
        if not passed:
            gate = False
    return bool(gate), details


def evaluate_actor(actor, frame: pd.DataFrame, profile) -> dict[str, float]:
    delta_levels = None
    if bool(profile.model.use_hybrid_overlay):
        delta_levels = np.asarray(profile.model.delta_levels, dtype=np.float32)
    return evaluate_split(
        actor,
        frame,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        action_levels=action_levels_for_profile(profile),
        observation_window=int(profile.model.observation_window),
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=bool(profile.model.use_hybrid_overlay),
        delta_levels=delta_levels,
        feature_columns=feature_columns_for_packs(profile.feature.packs, profile_name=profile.name),
        mode_router_config=profile.mode_router if bool(profile.mode_router.enabled) else None,
        position_mapping=profile.model.position_mapping,
    )


def metric_row(fold: str, split: str, metrics: dict[str, float]) -> dict[str, Any]:
    return {
        'fold': fold,
        'split': split,
        'return_pct': float(metrics['return']),
        'sharpe': float(metrics['sharpe']),
        'max_dd_pct': float(metrics['max_dd']),
        'avg_pos': float(metrics['avg_pos']),
        'stress_avg_pos': float(metrics['stress_avg_pos']),
        'stress_days': float(metrics['stress_days']),
        'buy_and_hold_return_pct': float(metrics['buy_and_hold_return']),
        'relative_return_pct': float(metrics['relative_return']),
    }


def loss_below(value: float, threshold: float) -> float:
    if not math.isfinite(float(value)):
        return 1.0
    return max(0.0, float(threshold) - float(value)) / max(abs(float(threshold)), 1.0)


def loss_above(value: float, ceiling: float) -> float:
    if not math.isfinite(float(value)):
        return 1.0
    return max(0.0, float(value) - float(ceiling)) / max(abs(float(ceiling)), 1.0)


def screen_slice_loss(details: dict[str, Any]) -> float:
    loss = 0.0
    for item in details.values():
        if item['status'] == 'not_applicable_zero_samples':
            continue
        loss += loss_below(float(item['samples']), float(item['min_samples']))
        loss += loss_below(float(item['accuracy']) if item['accuracy'] is not None else float('nan'), float(item['min_accuracy']))
    return float(loss)


set_seed(SEED)
profile = load_profile(PROFILE_NAME)
source_summary = load_json(SOURCE_SUMMARY_PATH)
rebound_summary = load_json(REBOUND_SUMMARY_PATH)
wf2020_summary = load_json(WF2020_SUMMARY_PATH)
reference_metrics = load_reference_metrics(REFERENCE_METRICS_PATH)

wf2020_contract = load_contract(WF2020_TARGET_PATH, WF2020_TARGET_CARD_NAME)
rebound_contract = load_contract(REBOUND_TARGET_PATH, REBOUND_TARGET_CARD_NAME)
contract = pd.concat([wf2020_contract, rebound_contract], ignore_index=True, sort=False)
fold_order = {fold: idx for idx, fold in enumerate(REQUIRED_FOLDS)}
contract['_fold_order'] = contract['fold'].astype(str).map(fold_order).fillna(999).astype(int)
contract = contract.sort_values(['_fold_order', 'datetime', 'source_contract_id']).drop(columns=['_fold_order']).reset_index(drop=True)
contract.to_csv(CARD / 'combined_target_contract.csv', index=False)

init_qlib(QLIB_DATA_PATH)

fold_frames: dict[str, dict[str, Any]] = {}
coverage_rows: list[dict[str, Any]] = []
for fold, window in WALK_FORWARD_WINDOWS.items():
    train_targets, audit = target_fold_rows(contract, fold, window)
    dataset = build_dataset(
        train_seg=window['train'],
        valid_seg=window['valid'],
        test_seg=window['test'],
        ablate_vix=False,
        feature_packs=profile.feature.packs,
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, 'train')
    valid_df = segment_frame(dataset, 'valid')
    test_df = segment_frame(dataset, 'test')
    dataset_samples = 0
    if not train_targets.empty:
        bc_dataset = build_state_objective_bc_dataset_from_frame(train_df, targets=train_targets, profile=profile, rng_seed=SEED)
        dataset_samples = int(len(bc_dataset.actions))
    audit['bc_dataset_samples'] = dataset_samples
    audit['dropped_rows_before_dataset'] = int(max(audit['train_rows'] - dataset_samples, 0))
    audit['target_contract_gate'] = bool(audit['target_contract_gate'] and dataset_samples > 0)
    coverage_rows.append(audit)
    fold_frames[fold] = {
        'window': window,
        'train_targets': train_targets,
        'train_df': train_df,
        'valid_df': valid_df,
        'test_df': test_df,
        'audit': audit,
    }

coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'target_scope_audit.csv', index=False)
combined_target_gate = bool((coverage['target_contract_gate'].astype(bool)).all() and set(REQUIRED_FOLDS).issubset(set(contract['fold'].astype(str))))

config_rows = []
for item in WEIGHT_CONFIGS:
    cfg = item['sample_weight_config']
    train_cfg = item['train_config']
    row = {
        'config_name': item['config_name'],
        'complexity': item['complexity'],
        'description': item['description'],
        'train_steps': int(train_cfg.steps),
        'lr': float(train_cfg.lr),
        'batch_size': int(train_cfg.batch_size),
        **asdict(cfg),
    }
    config_rows.append(row)
pd.DataFrame(config_rows).to_csv(CARD / 'bc_weight_configs.csv', index=False)

fit_cache: dict[tuple[str, str], Any] = {}
screen_rows: list[dict[str, Any]] = []
fold_rows: list[dict[str, Any]] = []
slice_rows: list[dict[str, Any]] = []
action_rows: list[dict[str, Any]] = []

for item in WEIGHT_CONFIGS:
    config_name = item['config_name']
    complexity = int(item['complexity'])
    sample_weight_config = item['sample_weight_config']
    train_config = item['train_config']
    per_fold_accuracy: list[float] = []
    per_fold_samples: list[int] = []
    all_fold_gates: list[bool] = []
    all_slice_gates: list[bool] = []
    config_loss = 0.0
    selected_slice_min: dict[str, dict[str, Any]] = {
        key: {'samples': None, 'accuracy': None, 'gate': None} for key in REQUIRED_SLICE_RULES
    }

    for fold, bundle in fold_frames.items():
        audit = bundle['audit']
        train_targets = bundle['train_targets']
        train_df = bundle['train_df']
        base_fold_row = {
            'config_name': config_name,
            'complexity': complexity,
            'fold': fold,
            'target_fold': bundle['window']['target_fold'],
            'target_rows': int(audit['train_rows']),
            'test_overlap_rows': int(audit['test_overlap_rows']),
        }
        if train_targets.empty or audit['test_overlap_rows'] > TARGET_TEST_OVERLAP_MAX:
            row = {
                **base_fold_row,
                'samples': 0,
                'loss': math.nan,
                'accuracy': math.nan,
                'fold_gate': False,
                'before_accuracy': math.nan,
                'after_accuracy': math.nan,
                'after_loss': math.nan,
                'status': 'blocked_missing_or_leaking_targets',
            }
            fold_rows.append(row)
            all_fold_gates.append(False)
            config_loss += 1.0
            continue

        set_seed(SEED)
        fit = fit_state_objective_bc_actor_from_frame(
            train_df,
            targets=train_targets,
            profile=profile,
            train_config=train_config,
            sample_weight_config=sample_weight_config,
            rng_seed=SEED,
        )
        fit_cache[(config_name, fold)] = fit
        bc_dataset = build_state_objective_bc_dataset_from_frame(train_df, targets=train_targets, profile=profile, rng_seed=SEED)
        fold_eval = evaluate_bc_actor(fit.actor, bc_dataset.obs, bc_dataset.actions)
        fold_accuracy = float(fold_eval['accuracy'])
        fold_samples = int(fold_eval['samples'])
        fold_gate = bool(fold_samples > 0 and fold_accuracy >= FOLD_MIN_ACCURACY)
        per_fold_accuracy.append(fold_accuracy)
        per_fold_samples.append(fold_samples)
        all_fold_gates.append(fold_gate)
        config_loss += loss_below(fold_accuracy, FOLD_MIN_ACCURACY)

        fold_rows.append({
            **base_fold_row,
            'samples': fold_samples,
            'loss': float(fold_eval['loss']),
            'accuracy': fold_accuracy,
            'fold_gate': fold_gate,
            'min_accuracy': FOLD_MIN_ACCURACY,
            'before_accuracy': float(fit.metrics['before_accuracy']),
            'after_accuracy': float(fit.metrics['after_accuracy']),
            'after_loss': float(fit.metrics['after_loss']),
            'weight_min': float(fit.metrics.get('weight_min', math.nan)),
            'weight_max': float(fit.metrics.get('weight_max', math.nan)),
            'status': 'evaluated',
        })

        slices = evaluate_bc_actor_slice_fit(
            fit.actor,
            fit.obs,
            fit.actions,
            bc_dataset.metadata,
            slice_cols=SLICE_COLS,
        )
        slice_gate, details = slice_gate_details(slices)
        all_slice_gates.append(slice_gate)
        config_loss += screen_slice_loss(details)
        for row in slices.to_dict('records'):
            row = dict(row)
            rule = REQUIRED_SLICE_RULES.get(str(row['slice']), {'min_samples': 0, 'min_accuracy': None, 'required': False})
            detail = details.get(str(row['slice']), {})
            row.update({
                'config_name': config_name,
                'complexity': complexity,
                'fold': fold,
                'min_samples': int(rule.get('min_samples', 0)),
                'min_accuracy': rule.get('min_accuracy'),
                'required_or_present': bool(detail.get('required', False)),
                'slice_gate': bool(detail.get('passed', True)),
                'slice_status': str(detail.get('status', 'informational')),
            })
            slice_rows.append(row)
        for slice_name, detail in details.items():
            current = selected_slice_min[slice_name]
            acc = detail['accuracy']
            if detail['status'] == 'not_applicable_zero_samples':
                continue
            if current['accuracy'] is None or (acc is not None and acc < current['accuracy']):
                selected_slice_min[slice_name] = {
                    'samples': int(detail['samples']),
                    'accuracy': float(acc) if acc is not None and math.isfinite(float(acc)) else math.nan,
                    'gate': bool(detail['passed']),
                }

        dist = fit.action_distribution.copy()
        dist['config_name'] = config_name
        dist['complexity'] = complexity
        dist['fold'] = fold
        action_rows.extend(dist.to_dict('records'))

    weighted_accuracy = float(np.average(per_fold_accuracy, weights=per_fold_samples)) if per_fold_accuracy and sum(per_fold_samples) > 0 else math.nan
    min_fold_accuracy = float(min(per_fold_accuracy)) if per_fold_accuracy else math.nan
    global_accuracy_gate = bool(math.isfinite(weighted_accuracy) and weighted_accuracy >= GLOBAL_MIN_ACCURACY)
    fold_accuracy_gate = bool(all_fold_gates and all(all_fold_gates) and math.isfinite(min_fold_accuracy) and min_fold_accuracy >= FOLD_MIN_ACCURACY)
    slice_gate = bool(all_slice_gates and all(all_slice_gates))
    screen_gate = bool(combined_target_gate and global_accuracy_gate and fold_accuracy_gate and slice_gate)
    row = {
        'config_name': config_name,
        'complexity': complexity,
        'description': item['description'],
        'folds': len(per_fold_accuracy),
        'required_folds': len(REQUIRED_FOLDS),
        'required_fold_coverage': bool(combined_target_gate),
        'total_samples': int(sum(per_fold_samples)),
        'weighted_accuracy': weighted_accuracy,
        'min_fold_accuracy': min_fold_accuracy,
        'global_min_accuracy': GLOBAL_MIN_ACCURACY,
        'fold_min_accuracy': FOLD_MIN_ACCURACY,
        'global_accuracy_gate': global_accuracy_gate,
        'fold_accuracy_gate': fold_accuracy_gate,
        'slice_gate': slice_gate,
        'bc_screen_gate': screen_gate,
        'gate_loss': float(config_loss),
        'train_steps': int(item['train_config'].steps),
        'lr': float(item['train_config'].lr),
        'batch_size': int(item['train_config'].batch_size),
        'promotion_allowed': False,
        'training_allowed_from_this_card': False,
    }
    for slice_name, detail in selected_slice_min.items():
        row[f'{slice_name}_min_samples'] = detail['samples']
        row[f'{slice_name}_min_accuracy'] = detail['accuracy']
        row[f'{slice_name}_gate'] = detail['gate'] if detail['gate'] is not None else True
    screen_rows.append(row)

screen = pd.DataFrame(screen_rows).sort_values(['bc_screen_gate', 'gate_loss', 'complexity'], ascending=[False, True, True]).reset_index(drop=True)
fold_metrics = pd.DataFrame(fold_rows)
slice_metrics = pd.DataFrame(slice_rows)
action_distribution = pd.DataFrame(action_rows)
screen.to_csv(CARD / 'bc_weight_screen.csv', index=False)
fold_metrics.to_csv(CARD / 'fold_bc_metrics.csv', index=False)
slice_metrics.to_csv(CARD / 'state_slice_bc_metrics.csv', index=False)
action_distribution.to_csv(CARD / 'bc_action_distribution.csv', index=False)

passed = screen[screen['bc_screen_gate'].astype(bool)]
selected = (passed.iloc[0] if not passed.empty else screen.iloc[0]).to_dict()
selected_config = str(selected['config_name'])
selected_screen_gate = bool(selected['bc_screen_gate'])
selected_fold_metrics = fold_metrics[fold_metrics['config_name'].eq(selected_config)].copy()
selected_slice_metrics = slice_metrics[slice_metrics['config_name'].eq(selected_config)].copy()
selected_slice_metrics.to_csv(CARD / 'selected_state_slice_bc_metrics.csv', index=False)

replay_rows: list[dict[str, Any]] = []
fold_score_rows: list[dict[str, Any]] = []
baseline_rows: list[dict[str, Any]] = []

for fold, bundle in fold_frames.items():
    window = bundle['window']
    train_targets = bundle['train_targets']
    ref = reference_row(reference_metrics, fold)
    audit = bundle['audit']
    base_score = {
        'fold': fold,
        'target_fold': window['target_fold'],
        'selected_config': selected_config,
        **audit,
        **ref,
    }
    fit = fit_cache.get((selected_config, fold))
    if fit is None or train_targets.empty or audit['test_overlap_rows'] > TARGET_TEST_OVERLAP_MAX:
        base_score.update({
            'status': 'skipped_missing_selected_fit_or_targets',
            'candidate_gate': False,
            'slice_gate': False,
            'fold_gate': False,
            'failed_checks': 'missing_selected_fit_or_targets',
        })
        fold_score_rows.append(base_score)
        baseline_rows.append(base_score)
        continue

    valid_metrics = evaluate_actor(fit.actor, bundle['valid_df'], profile)
    test_metrics = evaluate_actor(fit.actor, bundle['test_df'], profile)
    replay_rows.append(metric_row(fold, 'valid', valid_metrics))
    replay_rows.append(metric_row(fold, 'walk_forward_test', test_metrics))

    selected_fold_slices = selected_slice_metrics[selected_slice_metrics['fold'].eq(fold)].copy()
    slice_ok, slice_detail = slice_gate_details(selected_fold_slices)
    candidate_gate = bool(test_metrics['max_dd'] < TARGET_MAX_DD_PCT and test_metrics['relative_return'] >= TARGET_RELATIVE_RETURN_PP)
    failed_checks: list[str] = []
    if test_metrics['max_dd'] >= TARGET_MAX_DD_PCT:
        failed_checks.append('test_max_dd')
    if test_metrics['relative_return'] < TARGET_RELATIVE_RETURN_PP:
        failed_checks.append('test_relative_return')
    if not slice_ok:
        failed_checks.append('slice_gate')
    fold_gate = bool(candidate_gate and slice_ok)
    model_vs_ref_return_delta = None if ref['reference_return_pct'] is None else float(test_metrics['return'] - ref['reference_return_pct'])
    model_vs_ref_dd_delta = None if ref['reference_max_dd_pct'] is None else float(test_metrics['max_dd'] - ref['reference_max_dd_pct'])
    qlib_reference_drawdown_gate = bool(ref['reference_max_dd_pct'] is not None and ref['reference_max_dd_pct'] < TARGET_MAX_DD_PCT)
    score = {
        **base_score,
        'status': 'evaluated',
        'bc_after_accuracy': float(selected_fold_metrics[selected_fold_metrics['fold'].eq(fold)]['after_accuracy'].iloc[0]),
        'valid_return_pct': float(valid_metrics['return']),
        'valid_relative_return_pp': float(valid_metrics['relative_return']),
        'valid_max_dd_pct': float(valid_metrics['max_dd']),
        'test_return_pct': float(test_metrics['return']),
        'test_relative_return_pp': float(test_metrics['relative_return']),
        'test_max_dd_pct': float(test_metrics['max_dd']),
        'test_avg_pos': float(test_metrics['avg_pos']),
        'slice_gate': slice_ok,
        'slice_gate_detail': json.dumps(json_safe(slice_detail), sort_keys=True),
        'candidate_gate': candidate_gate,
        'fold_gate': fold_gate,
        'qlib_reference_drawdown_gate': qlib_reference_drawdown_gate,
        'model_vs_reference_return_delta_pct': model_vs_ref_return_delta,
        'model_vs_reference_max_dd_delta_pct': model_vs_ref_dd_delta,
        'failed_checks': ','.join(failed_checks),
    }
    fold_score_rows.append(score)
    baseline_rows.append(score)

replay_metrics = pd.DataFrame(replay_rows)
fold_scores = pd.DataFrame(fold_score_rows)
baseline_comparison = pd.DataFrame(baseline_rows)
replay_metrics.to_csv(CARD / 'fold_replay_metrics.csv', index=False)
fold_scores.to_csv(CARD / 'fold_gate_scores.csv', index=False)
baseline_comparison.to_csv(CARD / 'soxlbacktest_qlib_baseline_comparison.csv', index=False)

required_fold_names = list(WALK_FORWARD_WINDOWS)
evaluated = fold_scores[fold_scores['status'].eq('evaluated')].copy() if not fold_scores.empty else pd.DataFrame()
missing_folds = [fold for fold in required_fold_names if fold not in set(evaluated.get('fold', pd.Series(dtype=str)))]
failed_folds = fold_scores.loc[~fold_scores.get('fold_gate', pd.Series(False, index=fold_scores.index)).fillna(False).astype(bool), 'fold'].tolist() if not fold_scores.empty else required_fold_names
fold_gate_pass_count = int(fold_scores.get('fold_gate', pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if not fold_scores.empty else 0
candidate_gate_pass_count = int(fold_scores.get('candidate_gate', pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if not fold_scores.empty else 0
slice_gate_pass_count = int(fold_scores.get('slice_gate', pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if not fold_scores.empty else 0
full_replay_gate = bool(not missing_folds and fold_gate_pass_count == len(required_fold_names))

if selected_screen_gate and full_replay_gate:
    decision = 'rebound_state_factor_target_repair_bc_replay_passed_no_promotion'
    next_mechanism = 'train_full_retention_candidate_with_release16_cb99_warmstart_then_run_ytd_dashboard_and_actor_retention_gates'
    primary_blocker = 'full_retention_training_required_after_target_repair'
elif selected_screen_gate:
    decision = 'rebound_state_factor_target_repair_screen_ready_for_checkpoint_no_promotion'
    next_mechanism = 'run_release16_cb99_foldlocal_checkpoint_or_ppo_screen_before_actor_retention'
    primary_blocker = 'target_revision_ready'
else:
    decision = 'rebound_state_factor_target_repair_screen_failed_no_training'
    next_mechanism = 'revise_pre2020_factor_semantics_or_state_weighting_before_checkpoint_training'
    primary_blocker = 'wf2023_slice_fit_failed_after_target_repair'

metrics_for_frontier = []
if not replay_metrics.empty:
    for row in replay_metrics[replay_metrics['split'].eq('walk_forward_test')].to_dict('records'):
        metrics_for_frontier.append({
            'window': row['fold'],
            'split_type': 'walk_forward_test',
            'return_pct': row['return_pct'],
            'max_dd_pct': row['max_dd_pct'],
            'relative_return_pct': row['relative_return_pct'],
            'avg_pos': row['avg_pos'],
        })

sidecars = {
    'combined_target_contract': str((CARD / 'combined_target_contract.csv').relative_to(ROOT)),
    'target_scope_audit': str((CARD / 'target_scope_audit.csv').relative_to(ROOT)),
    'bc_weight_configs': str((CARD / 'bc_weight_configs.csv').relative_to(ROOT)),
    'bc_weight_screen': str((CARD / 'bc_weight_screen.csv').relative_to(ROOT)),
    'fold_bc_metrics': str((CARD / 'fold_bc_metrics.csv').relative_to(ROOT)),
    'state_slice_bc_metrics': str((CARD / 'state_slice_bc_metrics.csv').relative_to(ROOT)),
    'selected_state_slice_bc_metrics': str((CARD / 'selected_state_slice_bc_metrics.csv').relative_to(ROOT)),
    'fold_replay_metrics': str((CARD / 'fold_replay_metrics.csv').relative_to(ROOT)),
    'fold_gate_scores': str((CARD / 'fold_gate_scores.csv').relative_to(ROOT)),
    'bc_action_distribution': str((CARD / 'bc_action_distribution.csv').relative_to(ROOT)),
    'baseline_comparison': str((CARD / 'soxlbacktest_qlib_baseline_comparison.csv').relative_to(ROOT)),
}

selected_slices_by_fold = {}
for fold in required_fold_names:
    rows = selected_slice_metrics[selected_slice_metrics['fold'].eq(fold)].copy()
    selected_slices_by_fold[fold] = rows[['slice', 'samples', 'accuracy', 'slice_gate', 'slice_status']].to_dict('records') if not rows.empty else []

summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'profile': PROFILE_NAME,
    'source_card': SOURCE_CARD_NAME,
    'direct_comparison_baseline': SOURCE_CARD_NAME,
    'canonical_baseline': 'model_retention_frontier_v1',
    'reference_baseline': f'{REFERENCE_EXPERIMENT}:{REFERENCE_CANDIDATE}',
    'source_target_contracts': [REBOUND_TARGET_CARD_NAME, WF2020_TARGET_CARD_NAME],
    'source_target_paths': [str(REBOUND_TARGET_PATH.relative_to(ROOT)), str(WF2020_TARGET_PATH.relative_to(ROOT))],
    'source_decisions': {
        SOURCE_CARD_NAME: source_summary.get('decision'),
        REBOUND_TARGET_CARD_NAME: rebound_summary.get('decision'),
        WF2020_TARGET_CARD_NAME: wf2020_summary.get('decision'),
    },
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'target_contract', 'bc_sample_weighting'],
    'target_contract_gate': combined_target_gate,
    'bc_screen_gate': selected_screen_gate,
    'bc_readiness_gate': False,
    'next_training_allowed_after_target_revision': bool(selected_screen_gate),
    'checkpoint_screen_allowed_next': bool(selected_screen_gate),
    'ppo_screen_allowed_next': bool(selected_screen_gate),
    'training_allowed': False,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'next_mechanism': next_mechanism,
    'primary_blocker': primary_blocker,
    'recommended_primary_blocker': primary_blocker,
    'selected_config': selected_config,
    'selected_screen_gate': selected_screen_gate,
    'selected_weighted_accuracy': selected.get('weighted_accuracy'),
    'selected_min_fold_accuracy': selected.get('min_fold_accuracy'),
    'selected_gate_loss': selected.get('gate_loss'),
    'screen_rows': screen.to_dict('records'),
    'selected_slice_metrics_by_fold': selected_slices_by_fold,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_relative_return_by_window_pp': {fold: TARGET_RELATIVE_RETURN_PP for fold in required_fold_names},
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'drawdown_gate_operator': '<',
    'required_wf_windows': required_fold_names,
    'evaluated_folds': evaluated['fold'].tolist() if not evaluated.empty else [],
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'fold_gate_pass_count': fold_gate_pass_count,
    'candidate_gate_pass_count': candidate_gate_pass_count,
    'slice_gate_pass_count': slice_gate_pass_count,
    'full_replay_gate': full_replay_gate,
    'metrics': metrics_for_frontier,
    'target_scope_audit': coverage.to_dict('records'),
    'sidecars': sidecars,
}
selection = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'created_at': summary['generated_at'],
    'profile': PROFILE_NAME,
    'source_card': SOURCE_CARD_NAME,
    'reference_baseline': summary['reference_baseline'],
    'selected_config': selected_config,
    'target_contract_gate': combined_target_gate,
    'bc_screen_gate': selected_screen_gate,
    'next_training_allowed_after_target_revision': bool(selected_screen_gate),
    'checkpoint_screen_allowed_next': bool(selected_screen_gate),
    'ppo_screen_allowed_next': bool(selected_screen_gate),
    'training_allowed': False,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'primary_blocker': primary_blocker,
    'next_mechanism': next_mechanism,
    'required_wf_windows': required_fold_names,
    'evaluated_folds': summary['evaluated_folds'],
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'sidecars': sidecars,
}
write_json(CARD / 'summary.json', summary)
write_json(CARD / 'selection.json', selection)

conclusion = (
    f"# {EXPERIMENT}\n\n"
    f"- Decision: `{decision}`.\n"
    f"- Selected BC config: `{selected_config}` with weighted accuracy `{selected.get('weighted_accuracy'):.3f}` and min fold accuracy `{selected.get('min_fold_accuracy'):.3f}`.\n"
    f"- Target contract gate: `{combined_target_gate}`; BC screen gate: `{selected_screen_gate}`; replay fold gates: `{fold_gate_pass_count}/{len(required_fold_names)}`.\n"
    f"- Missing folds: `{missing_folds}`; failed folds: `{failed_folds}`.\n"
    f"- Next: `{next_mechanism}`.\n"
    "- No actor is retained or promoted by this diagnostic notebook.\n"
)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')

display(coverage)
display(screen)
display(fold_scores)
display(Markdown(conclusion))
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[57802:MainThread](2026-06-14 21:03:17,073) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[57802:MainThread](2026-06-14 21:03:17,574) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[57802:MainThread](2026-06-14 21:03:17,575) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[57802:MainThread](2026-06-14 21:03:18,439) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

,fold,target_fold,source_contract_ids,fold_rows,train_rows,valid_overlap_rows,test_overlap_rows,train_start,train_end,valid_start,...,test_start,test_end,release_gate_rows,factor_rebound_rows,rebound_cluster_rows,downtrend_rows,drawdown_control_rows,target_contract_gate,bc_dataset_samples,dropped_rows_before_dataset
0,wf_2020,wf_2020_train,cap10_wf2020_soxlbacktest_target_contract_v1,2470,2218,252,0,2010-03-11,2018-12-31,2019-01-01,...,2020-01-01,2020-12-31,38,0,38,768,768,True,1904,314
1,wf_2022,wf_2022_train,rebound_state_factor_target_contract_v1,505,253,252,0,2010-03-11,2020-12-31,2021-01-01,...,2022-01-01,2022-12-31,8,8,12,118,0,True,252,1
2,wf_2023_2024,wf_2023_2024_train,rebound_state_factor_target_contract_v1,756,756,0,0,2010-03-11,2022-12-31,2023-01-01,...,2024-01-01,2024-12-31,11,11,14,478,0,True,755,1


,config_name,complexity,description,folds,required_folds,required_fold_coverage,total_samples,weighted_accuracy,min_fold_accuracy,global_min_accuracy,...,factor_rebound_state_gate,is_rebound_cluster_min_samples,is_rebound_cluster_min_accuracy,is_rebound_cluster_gate,is_downtrend_state_min_samples,is_downtrend_state_min_accuracy,is_downtrend_state_gate,drawdown_control_state_min_samples,drawdown_control_state_min_accuracy,drawdown_control_state_gate
0,release16_rebound6_downtrend2_cb99,4,"Release-heavy, rebound-heavy, class-balanced w...",3,3,True,2911,0.751975,0.727416,0.7,...,True,14,0.714286,True,477,0.821803,True,626,0.921725,True
1,release9_rebound4_downtrend2_steps1000,3,Longer optimization sanity check without class...,3,3,True,2911,0.782892,0.752318,0.7,...,False,14,0.357143,False,477,0.840671,True,626,0.936102,True
2,release12_rebound4_downtrend2_steps600,2,"Slightly stronger release weighting, no class ...",3,3,True,2911,0.737891,0.666225,0.7,...,False,14,0.142857,False,477,0.775681,True,626,0.916933,True
3,release10_rebound4_downtrend2_steps600,1,Prior moderate release-gate weighting from cap...,3,3,True,2911,0.731020,0.660927,0.7,...,False,14,0.142857,False,477,0.769392,True,626,0.912141,True


,fold,target_fold,selected_config,source_contract_ids,fold_rows,train_rows,valid_overlap_rows,test_overlap_rows,train_start,train_end,...,test_max_dd_pct,test_avg_pos,slice_gate,slice_gate_detail,candidate_gate,fold_gate,qlib_reference_drawdown_gate,model_vs_reference_return_delta_pct,model_vs_reference_max_dd_delta_pct,failed_checks
0,wf_2020,wf_2020_train,release16_rebound6_downtrend2_cb99,cap10_wf2020_soxlbacktest_target_contract_v1,2470,2218,252,0,2010-03-11,2018-12-31,...,22.261651,0.327037,True,"{""drawdown_control_state"": {""accuracy"": 0.9217...",False,False,True,36.028106,9.476444,"test_max_dd,test_relative_return"
1,wf_2022,wf_2022_train,release16_rebound6_downtrend2_cb99,rebound_state_factor_target_contract_v1,505,253,252,0,2010-03-11,2020-12-31,...,21.310936,0.166648,True,"{""drawdown_control_state"": {""accuracy"": null, ...",False,False,True,-0.280958,2.796351,test_max_dd
2,wf_2023_2024,wf_2023_2024_train,release16_rebound6_downtrend2_cb99,rebound_state_factor_target_contract_v1,756,756,0,0,2010-03-11,2022-12-31,...,8.845828,0.114725,True,"{""drawdown_control_state"": {""accuracy"": null, ...",True,True,True,-2.036308,7.784025,


# rebound_state_factor_target_repair_screen_v1

- Decision: `rebound_state_factor_target_repair_screen_ready_for_checkpoint_no_promotion`.
- Selected BC config: `release16_rebound6_downtrend2_cb99` with weighted accuracy `0.752` and min fold accuracy `0.727`.
- Target contract gate: `True`; BC screen gate: `True`; replay fold gates: `1/3`.
- Missing folds: `[]`; failed folds: `['wf_2020', 'wf_2022']`.
- Next: `run_release16_cb99_foldlocal_checkpoint_or_ppo_screen_before_actor_retention`.
- No actor is retained or promoted by this diagnostic notebook.


{'experiment': 'rebound_state_factor_target_repair_screen_v1',
 'decision': 'rebound_state_factor_target_repair_screen_ready_for_checkpoint_no_promotion',
 'generated_at': '2026-06-14T13:03:55.762908+00:00',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'source_card': 'rebound_state_factor_bc_replay_v1',
 'direct_comparison_baseline': 'rebound_state_factor_bc_replay_v1',
 'canonical_baseline': 'model_retention_frontier_v1',
 'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02',
 'source_target_contracts': ['rebound_state_factor_target_contract_v1',
  'cap10_wf2020_soxlbacktest_target_contract_v1'],
 'source_target_paths': ['runtime/model_selection/card/rebound_state_factor_target_contract_v1/selected_target_contract.csv',
  'runtime/model_selection/card/cap10_wf2020_soxlbacktest_target_contract_v1/soxlbacktest_wf2020_target_contract.csv'],
 'source_decisions': {'rebound_state_factor_bc_replay_v1': 'rebound_state_factor_bc_replay_partial_missing_wf_